# mini_gpt_torch : Pre-training a 124M-Parameter Mini GPT

The GPT-2 Small model contains approximately 124 million parameters and consists of 12 Transformer layers, 12 Attention heads, and 768-dimensional embeddings. Training this model from scratch on a suitable GPU can take several hours or more, depending on the hardware, dataset size, and training configurations. While most practitioners use pre-trained checkpoints instead, implementing and training an instance from scratch provides a much deeper understanding of the inner workings of the models upon which modern products are built.


### Learning Objectives
- Fully implement the GPT-2 architecture (~124M parameters) from scratch, including token embeddings, positional embeddings, Transformer blocks, and the language model head.
- Train a GPT model on a text corpus using the next-token prediction objective and cross-entropy loss.
- Implement autoregressive text generation using temperature sampling along with top-k and top-p filtering.
- Monitor training loss curves and evaluate whether the model has learned coherent linguistic patterns.


### Problem Statement
You know what a Transformer is, you have seen its diagrams, you can cite "Attention Is All You Need," and you can draw boxes labeled Multi-Head Attention on a whiteboard.

However, none of these alone mean you know precisely what happens inside the model during text generation.

The GPT-2 Small model, using weight tying, contains 124,402,944 parameters. Every single parameter value is determined through iterative execution of a training loop: running the forward pass, computing the loss, executing the backward pass, and updating the weights. This model comprises 12 Transformer blocks, 12 attention heads per block, an embedding dimension of 768, and a vocabulary size of 50,257 tokens. Each time the model generates a token, its parameters participate in a cascade of matrix operations—a sequence of computations that ingests a series of token IDs and outputs a probability distribution over the next token.

Without implementing this process yourself, the model largely remains a black box. You can query APIs or fine-tune checkpoints, but when issues arise—such as hallucinations, degenerate repetitions, or failure to follow instructions—you lack the precise mental model required to diagnose the underlying causes.

### Core Concept
#### GPT Architecture
GPT is an autoregressive language model. The term "autoregressive" means that the model generates one token at a time, where each token depends on all previously generated tokens. The architecture consists of a stack of Transformer decoder blocks.

The complete computational path from token IDs to next-token probabilities proceeds as follows:

1. Token IDs enter the model. Input shape is `(batch_size, seq_len)`.
2. Token embedding lookup is executed. Each ID is mapped to a 768-dimensional vector. Output shape is `(batch_size, seq_len, 768)`.
3. Position embedding lookup is executed. Each position index ($0, 1, 2, \dots$) is mapped to a 768-dimensional vector, with output shape matching the previous step.
4. Token embeddings and position embeddings are element-wise summed.
5. The representation passes through 12 Transformer blocks.
6. A final LayerNorm is applied.
7. A linear projection maps the embedding dimension to the vocabulary size. Output shape is `(batch_size, seq_len, vocab_size)`.
8. Softmax is applied to convert logits into probabilities.

This is the entire model architecture: no convolutions, no recurrence. The model relies exclusively on embeddings, Attention, feed-forward networks, and LayerNorm stacked iteratively.




In [1]:
"""
Mini GPT — PyTorch implementation exercise.

ALLOWED PyTorch APIs:
    torch tensor operations, torch.nn.Parameter, torch.nn.Linear, torch.nn.Embedding,
    torch.nn.Module, torch.autograd, torch.optim.

BANNED PyTorch APIs (you must implement these yourself):
    torch.nn.LayerNorm, torch.nn.functional.layer_norm,
    torch.nn.MultiheadAttention, torch.nn.functional.scaled_dot_product_attention,
    torch.nn.Transformer / nn.TransformerEncoderLayer / nn.TransformerDecoderLayer,
    torch.nn.functional.softmax, torch.softmax, Tensor.softmax,
    torch.nn.functional.cross_entropy, torch.nn.functional.log_softmax,
    torch.nn.CrossEntropyLoss.

Gradients are handled entirely by autograd — you never write a backward pass. Every
forward pass you write must therefore stay differentiable: build outputs from tensor
operations on the inputs, and never call .detach(), .item(), .numpy(), or wrap
anything in torch.no_grad() except where a docstring explicitly says so.

All tensors are float32 unless stated otherwise, except `token_ids`, which is
torch.long. Do not hardcode dtypes or devices inside forward passes: derive them from
the incoming tensors, so the same code runs unchanged in float64.
"""


'\nMini GPT — PyTorch implementation exercise.\n\nALLOWED PyTorch APIs:\n    torch tensor operations, torch.nn.Parameter, torch.nn.Linear, torch.nn.Embedding,\n    torch.nn.Module, torch.autograd, torch.optim.\n\nBANNED PyTorch APIs (you must implement these yourself):\n    torch.nn.LayerNorm, torch.nn.functional.layer_norm,\n    torch.nn.MultiheadAttention, torch.nn.functional.scaled_dot_product_attention,\n    torch.nn.Transformer / nn.TransformerEncoderLayer / nn.TransformerDecoderLayer,\n    torch.nn.functional.softmax, torch.softmax, Tensor.softmax,\n    torch.nn.functional.cross_entropy, torch.nn.functional.log_softmax,\n    torch.nn.CrossEntropyLoss.\n\nGradients are handled entirely by autograd — you never write a backward pass. Every\nforward pass you write must therefore stay differentiable: build outputs from tensor\noperations on the inputs, and never call .detach(), .item(), .numpy(), or wrap\nanything in torch.no_grad() except where a docstring explicitly says so.\n\nAll 

In [1]:
import torch
import torch.nn as nn
import math
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device = {device}")

GPU is available: NVIDIA GeForce RTX 5060 Laptop GPU
device = cuda


The Transformer architecture is inherently permutation invariant. By adding `pos_embed` to `token_embed`, we inject information about the positional location of each token in the sentence into the model.

**How (Broadcasting):** The `tok_emb` tensor has dimensions `(batch_size, seq_len, embed_dim)`, and `pos_emb` has dimensions `(seq_len, embed_dim)`. PyTorch automatically broadcasts `pos_emb` across the first dimension (batch) to match the dimensions of `tok_emb` for addition.

**Device Management:** To avoid `RuntimeError` (device mismatch between GPU/CPU), we used `token_ids.device` so that the position tensor (`pos_ids`) is created on the exact same memory device where `token_ids` resides.


In [2]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len):
        """
        Token and positional embedding layer.

        Args:
            vocab_size (int): Size of the vocabulary.
            embed_dim (int): Dimensionality of embedding vectors.
            max_seq_len (int): Maximum sequence length supported by positional embeddings.

        Attributes:
            token_embed (nn.Embedding): Token embedding table. Weight shape: (vocab_size, embed_dim)
            pos_embed (nn.Embedding): Positional embedding table. Weight shape: (max_seq_len, embed_dim)
        """
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        """
        Computes combined token and positional embeddings for input token sequences.

        Args:
            token_ids (torch.Tensor): Token indices, dtype torch.long.
                Shape: (batch_size, seq_len)

        Returns:
            torch.Tensor: Sum of token embeddings and the positional embeddings for
                positions 0..seq_len-1, broadcast across the batch.
                Shape: (batch_size, seq_len, embed_dim)
        """
        # input shape
        batch_size, seq_len = token_ids.shape
        
        # extract token
        tok_emb = self.token_embed(token_ids)
        
        # Creating position indices (0 to seq_len-1) and moving to the same device as the input
        pos_ids = torch.arange(seq_len, device=token_ids.device)
        pos_emb = self.pos_embed(pos_ids)
        
        return tok_emb + pos_emb


In [3]:
vocab_size = 100
embed_dim = 16
max_seq_len = 32
batch_size = 2
seq_len = 10

model = Embedding(vocab_size, embed_dim, max_seq_len).to(device)
print(model)

# random input
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
print(token_ids)

#Forward
output = model(token_ids)
# print(output)

assert output.shape == (batch_size, seq_len, embed_dim), f"Expected {(batch_size, seq_len, embed_dim)}, got {output.shape}"
assert output.device.type == device.type, "Output is not on the correct device"

loss = output.sum()
loss.backward()

print("Test Passed: Embedding layer is working correctly with GPU.")
print(f"Output shape: {output.shape}")
print(f"Device: {output.device}")


Embedding(
  (token_embed): Embedding(100, 16)
  (pos_embed): Embedding(32, 16)
)
tensor([[81, 93, 50, 89, 20, 37, 66, 27, 15, 83],
        [72,  3, 62, 84, 11, 14, 18, 85, 27, 92]], device='cuda:0')
Test Passed: Embedding layer is working correctly with GPU.
Output shape: torch.Size([2, 10, 16])
Device: cuda:0


The goal of Layer Normalization in neural networks, and especially in Transformers, is to stabilize the distribution of activations across layers.

The key reasons for its implementation are as follows:

**Preventing Internal Covariate Shift:**
As the network deepens, the distribution of inputs to each layer changes constantly, which slows down learning. Normalization ensures that the mean of each feature approaches 0 and its variance approaches 1.

**Scale & Shift Control:**
If we were to only normalize the data, the model's capacity to learn complex patterns might be limited. That is why we use learnable parameters, `gamma` (scale) and `beta` (shift), allowing the network to determine the optimal scale and mean for subsequent layers.

**Independence from Batch Size:**
Unlike Batch Normalization, which is dependent on the batch size, LayerNorm performs calculations independently for each sample along the feature dimension (`dim`). This property is critical for large language models and text generation (like Transformers), where sequence lengths or batch sizes may vary.



- `dim=-1`: Ensures that operations are performed exclusively along the last dimension (feature/embedding dimension). This means whether your input shape is `(batch, seq, dim)` or even `(batch, dim)`, the code works without modification.

- `keepdim=True`: Crucial. With this parameter, the output of `mean` and `var` retains the last dimension as `(..., 1)` instead of squeezing it out. This allows PyTorch to automatically broadcast the tensor across all features during normalization.
Stability: Adding `eps` to the variance before taking the square root prevents `NaN` errors in case the variance is zero.


In [4]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        """
        Layer Normalization across the feature dimension.

        Args:
            dim (int): Feature/embedding dimension to normalize.
            eps (float): Epsilon added to the variance for numerical stability.

        Attributes:
            gamma (nn.Parameter): Learnable scale, initialized to ones. Shape: (dim,)
            beta (nn.Parameter): Learnable shift, initialized to zeros. Shape: (dim,)
            eps (float): Stored epsilon value.
        """
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x):
        """
        Normalizes the last dimension of the input tensor and applies scale and shift.

        Args:
            x (torch.Tensor): Input tensor.
                Shape: (..., dim)

        Returns:
            torch.Tensor: Layer-normalized tensor with the same shape as the input.
                The mean and the biased variance are computed over the last axis only.
                Shape: (..., dim)
        """
        # Compute the mean along the last dimension (feature dimension)
        # `keepdim=True` is necessary to preserve dimensions for broadcasting
        mean = x.mean(dim=-1, keepdim=True)
        
        # Compute the variance (biased) along the last dimension
        # Variance = Mean of squared differences from the mean
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        
        # Normalization
        # Add eps to prevent division by zero (numerical stability)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # 4. Scale and Shift
        return x_norm * self.gamma + self.beta


In [5]:
ln = LayerNorm(dim=6).to(device)
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
                   [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]], device=device)
output = ln(x)

print("\nOutput values:\n", output)
print("\nCheck if output is on device:", output.device)


Output values:
 tensor([[[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
         [-0.4472, -0.4472, -0.4472, -0.4472, -0.4472,  2.2360]]],
       device='cuda:0', grad_fn=<AddBackward0>)

Check if output is on device: cuda:0


### Multi-Head Self-Attention Implementation

This implementation represents the core of the Transformer. The logic and implementation details are as follows:

#### Purpose
*   **Projection (Q, K, V):** The input `x` (containing sequence information) is projected into three distinct spaces to simultaneously extract "Query," "Key," and "Value."
*   **Multi-Head:** Instead of a single large attention mechanism, we create multiple parallel smaller mechanisms (`num_heads`). This allows the model to attend to different relational dimensions between tokens (e.g., syntactic relationships in one head and semantic relationships in another).
*   **Causal Masking:** Since GPT is a generative model, it must not "see" future tokens. We apply masking to suppress future information.

#### Implementation Steps
1.  **Projection:** Transform `(B, T, C)` into three separate tensors for Q, K, and V.
2.  **Reshape & Transpose:** Reshape to `(B, H, T, Head_dim)` to perform matrix operations independently for each "head."
3.  **Attention Scores:** Dot product of `Q` and `K` transposed (scaled by $1/\sqrt{d_k}$).
4.  **Causal Masking:** Add `float('-inf')` to positions above the main diagonal of the score matrix.
5.  **Softmax:** Normalize scores into probability weights.
6.  **Context:** Multiply normalized weights by `V`.
7.  **Concat & Output:** Concatenate heads and project back to original dimension `C`.
```python
# 4. Apply Causal Mask
if mask is not None:
# mask must be (T, T) with 0 for allowed and -inf for masked
att = att.masked_fill(mask == 0, float('-inf'))

# 5. Softmax and weight V
att = torch.nn.functional.softmax(att, dim=-1)
y = att @ v # (B, H, T, T) @ (B, H, T, Head_dim) -> (B, H, T, Head_dim)

# 6. Concatenate heads
y = y.transpose(1, 2).contiguous().view(B, T, C)

# 7. Final output
return self.W_out(y)


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        """
        Causal Multi-Head Attention module.

        Args:
            embed_dim (int): Total dimensionality of input and output features.
            num_heads (int): Number of parallel attention heads. Must divide embed_dim.

        Attributes:
            num_heads (int): Number of attention heads.
            head_dim (int): embed_dim // num_heads.
            W_q (nn.Linear): Query projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_k (nn.Linear): Key projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_v (nn.Linear): Value projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_out (nn.Linear): Output projection, no bias. Weight shape: (embed_dim, embed_dim)
        """
        super().__init__()
        assert embed_dim % num_heads == 0, f"embed_dim {embed_dim} not divisible by num_heads {num_heads}"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_out = nn.Linear(embed_dim, embed_dim, bias=False)
        for layer in (self.W_q, self.W_k, self.W_v, self.W_out):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)

    def forward(self, x, mask=None):
        """
        Multi-head projection, scaled dot-product attention with optional additive
        masking, and output projection. The softmax must be implemented by hand.

        Args:
            x (torch.Tensor): Input tensor.
                Shape: (batch_size, seq_len, embed_dim)
            mask (torch.Tensor, optional): Additive attention mask, added to the
                attention scores before the softmax. Allowed positions hold 0.0, masked
                positions hold a large negative value (see `causal_mask`).
                Shape: (seq_len, seq_len) or broadcastable to
                (batch_size, num_heads, seq_len, seq_len). Defaults to None (no masking).

        Returns:
            torch.Tensor: Attention output after the heads are recombined and passed
                through W_out.
                Shape: (batch_size, seq_len, embed_dim)
        """
        B, T, C = x.shape  # Batch, Seq_len, Embed_dim

        # Computing Projections (Q, K, V)
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # Reshaping and Transposing into Heads: (B, H, T, D)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Computing Scaled Dot-Product Attention
        # (B, H, T, D) @ (B, H, D, T) -> (B, H, T, T)
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale

        # Applying Additive Causal Mask
        if mask is not None:
            att = att + mask

        # Numerically Stable Hand-crafted Softmax along the last dimension
        att_max = torch.max(att, dim=-1, keepdim=True).values
        exp_att = torch.exp(att - att_max)
        att_weights = exp_att / torch.sum(exp_att, dim=-1, keepdim=True)

        # Multiplication by Values: (B, H, T, T) @ (B, H, T, D) -> (B, H, T, D)
        y = att_weights @ v

        # Concatenate Heads: (B, H, T, D) -> (B, T, H, D) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.W_out(y)


In [7]:

mha = MultiHeadAttention(embed_dim=64, num_heads=4)
sample_x = torch.randn(2, 8, 64) # batch=2, seq_len=8, embed_dim=64
out = mha(sample_x)

print("Output shape:", out.shape)

Output shape: torch.Size([2, 8, 64])


### Feed-Forward Network

In addition to the Attention mechanism, which identifies relationships between different tokens, Transformers require a space to process features within each token and learn more complex non-linear relationships.

*   **First Layer (fc1):** Expands the feature dimensions (ff_dim is typically 4x embed_dim) to provide the model with higher capacity to learn complex patterns.
*   **Activation Function (ReLU):** Adds non-linearity to the network (without it, the entire network would effectively be a simple linear layer).
*   **Second Layer (fc2):** Projects the dimensions back to the original size (embed_dim) to allow integration with the rest of the model.

---

### Operations

*   **First Layer (`self.fc1(x)`):** The layer weights with dimensions `(ff_dim, embed_dim)` are applied to the input. Given an input of shape `(B, T, embed_dim)`, PyTorch applies this to the last dimension, resulting in an output of `(B, T, ff_dim)`.
*   **Non-linearity (`F.relu`):** Sets negative values to zero and keeps positive values unchanged.
*   **Second Layer (`self.fc2(x)`):** Compresses the values from the larger `ff_dim` back to `embed_dim`, ensuring the final output has the same dimensions as the input `(B, T, embed_dim)`.

---
### Optimization and Architectural Notes

1. **nn.ReLU usage:**
   It is common practice in PyTorch to define layers (like `nn.ReLU`) in `__init__` and call them in `forward`. However, using the functional API (`F.relu`) for activation layers without learnable parameters is standard and widely used (e.g., in nanoGPT).

2. **Alignment with Original Architecture:**
   The original GPT architecture uses `GELU` instead of `ReLU`. If aiming for exact alignment with OpenAI's implementation, `torch.nn.functional.gelu(x)` is preferred. However, `ReLU` is sufficient for learning core concepts.

3. **Performance:**
   The implementation is efficient. PyTorch handles these operations effectively, including in-place memory management on the GPU.


In [8]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        """
        Position-wise Feed-Forward Network (MLP).

        Args:
            embed_dim (int): Model embedding feature dimension.
            ff_dim (int): Hidden feature dimension of the expansion layer.

        Attributes:
            fc1 (nn.Linear): Expansion layer. Weight shape: (ff_dim, embed_dim), bias: (ff_dim,)
            fc2 (nn.Linear): Contraction layer. Weight shape: (embed_dim, ff_dim), bias: (embed_dim,)
        """
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, embed_dim)
        for layer in (self.fc1, self.fc2):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        """
        Two-layer feed-forward transformation with a ReLU activation in between.

        Args:
            x (torch.Tensor): Input hidden states.
                Shape: (..., embed_dim)

        Returns:
            torch.Tensor: Transformed features, projected up to ff_dim and back down.
                Shape: (..., embed_dim)
        """
        # Transforming `embed_dim` to `ff_dim`
        x = self.fc1(x)
        
        # Non-linear Activation
        # OpenAi use `GELU` but we use `ReLU`
        x = torch.nn.functional.relu(x)
        
        # Contraction: Returning to `embed_dim`
        x = self.fc2(x)
        
        return x

### TransformerBlock Architecture

The `TransformerBlock` is the fundamental, repeatable building block of Transformer-based models (such as GPT). It relies on two key mechanisms:

#### 1. Dual Processing Mechanisms
* **Self-Attention:** Captures relationships between different tokens across the sequence dimension ($T$).
* **FeedForward Network (FFN):** Processes token-level features independently and introduces non-linear transformations.

#### 2. Residual Connections (Skip Connections)
* **Formulation:** $x = x + \text{SubLayer}(x)$
* **Purpose:** Mitigates the vanishing gradient problem in deep architectures by providing an unobstructed path for gradients to propagate back to earlier layers during backpropagation.

#### 3. Pre-Layer Normalization (Pre-LN)
* In modern GPT architectures, normalization is applied before each sub-layer:
  $$\text{SubLayer}(\text{LayerNorm}(x))$$
  instead of Post-LN:
  $$\text{LayerNorm}(x + \text{SubLayer}(x))$$
* **Purpose:** Stabilizes feature variance across layers, leading to more stable optimization dynamics and reducing sensitivity to learning rate warmup schedules.

---

### Data Flow in `forward`

The forward pass operates sequentially across two sub-blocks:

1. **Attention Sub-block:**
   * Normalize input features: $x_{\text{norm}} = \text{ln1}(x)$
   * Compute multi-head attention: $a = \text{attn}(x_{\text{norm}}, \text{mask})$
   * Apply first residual connection: $x = x + a$

2. **Feed-Forward Sub-block:**
   * Normalize updated features: $x_{\text{norm}} = \text{ln2}(x)$
   * Compute feed-forward projection: $f = \text{ffn}(x_{\text{norm}})$
   * Apply second residual connection: $x = x + f$


In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        """
        Pre-LayerNorm Transformer block.

        Args:
            embed_dim (int): Embedding feature dimension.
            num_heads (int): Number of attention heads.
            ff_dim (int): Intermediate feed-forward layer dimension.

        Attributes:
            ln1 (LayerNorm): Normalization applied before attention.
            attn (MultiHeadAttention): Causal self-attention sub-layer.
            ln2 (LayerNorm): Normalization applied before the feed-forward network.
            ffn (FeedForward): Feed-forward sub-layer.
        """
        super().__init__()
        self.ln1 = LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.ln2 = LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, ff_dim)

    def forward(self, x, mask=None):
        """
        Passes the input through the attention and feed-forward sub-layers, each with
        pre-normalization and a residual connection.

        Args:
            x (torch.Tensor): Input representation.
                Shape: (batch_size, seq_len, embed_dim)
            mask (torch.Tensor, optional): Additive causal mask forwarded to attention.
                Shape: (seq_len, seq_len) or broadcastable. Defaults to None.

        Returns:
            torch.Tensor: Output representation with the same shape as the input.
                Shape: (batch_size, seq_len, embed_dim)
        """
        # 1. Attention Sub-layer with Pre-LN and Residual Connection
        norm_x = self.ln1(x)
        attn_out = self.attn(norm_x, mask=mask)
        x = x + attn_out

        # 2. Feed-Forward Sub-layer with Pre-LN and Residual Connection
        norm_x = self.ln2(x)
        ffn_out = self.ffn(norm_x)
        x = x + ffn_out

        return x


In [10]:
block = TransformerBlock(embed_dim=64, num_heads=4, ff_dim=256)
sample_x = torch.randn(2, 8, 64)

out = block(sample_x)
print("Block output shape:", out.shape)  # torch.Size([2, 8, 64])

Block output shape: torch.Size([2, 8, 64])


### Causal Masking (Autoregressive Masking)

In autoregressive language models such as GPT, token generation follows a strict causal order: a token at index $i$ is only permitted to attend to previous tokens and itself ($j \le i$), with access to future tokens ($j > i$) explicitly blocked.

Using an **Additive Mask**, attention logits corresponding to disallowed future positions are populated with a large negative value ($-\infty$ or the minimum representable value of the given data type). When passed through the Softmax function ($e^{-\infty} \to 0$), the resulting attention weights for those future tokens evaluate strictly to zero:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

Using `torch.finfo(dtype).min` instead of a hardcoded constant (e.g., `-1e9`) guarantees numerical stability and precision across various precision formats (`float32`, `float16`, `bfloat16`, `float64`) without risking overflow or underflow.

---

### Implementation Steps

1. **Upper Triangular Boolean Mask:**
   Generate a square boolean matrix of shape `(seq_len, seq_len)` where the strictly upper triangular entries ($j > i$) are set to `True` using `torch.triu(..., diagonal=1)`.

2. **Tensor Initialization:**
   Initialize a tensor of zeros matching the target `dtype` and `device`.

3. **Masked Fill:**
   Fill the upper triangular positions where the condition holds with `torch.finfo(dtype).min`.


In [11]:
def causal_mask(seq_len, dtype=torch.float32, device=None):
    """
    Builds the additive causal (autoregressive) attention mask.

    Args:
        seq_len (int): Sequence length.
        dtype (torch.dtype): Data type of the returned mask. Defaults to torch.float32.
        device (torch.device, optional): Device of the returned mask. Defaults to None (CPU).

    Returns:
        torch.Tensor: Square additive mask whose entry [i, j] is 0.0 when position i is
            allowed to attend to position j (j <= i) and a large negative value
            otherwise. Use torch.finfo(dtype).min rather than a hardcoded constant so
            the mask stays valid in float64.
            Shape: (seq_len, seq_len)
    """
    # make matrix zero
    mask = torch.zeros((seq_len, seq_len), dtype=dtype, device=device)

    # find type min value
    min_val = torch.finfo(dtype).min

    # Locate positions strictly above the main diagonal
    # `diagonal=1` targets the strictly upper triangular region
    upper_tri_indices = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)

    # Substitute large negative values into future positions.
    mask = mask.masked_fill(upper_tri_indices, min_val)

    return mask

In [12]:
print(seq_len := 5)
mask = torch.zeros((seq_len, seq_len), dtype=torch.float32, device=device)
print(mask)
min_val = torch.finfo(torch.float32).min
print(min_val)
upper_tri_indices = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)
print(upper_tri_indices)
mask = mask.masked_fill(upper_tri_indices, min_val)
print(mask)



5
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], device='cuda:0')
-3.4028234663852886e+38
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]], device='cuda:0')
tensor([[ 0.0000e+00, -3.4028e+38, -3.4028e+38, -3.4028e+38, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00, -3.4028e+38, -3.4028e+38, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, -3.4028e+38, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00, -3.4028e+38],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],
       device='cuda:0')


## `calss MiniGPT`

### `forward` Method

#### Purpose and Operations:
* **Token & Positional Embeddings:** Convert discrete token IDs into dense vectors by summing semantic token embeddings and positional embeddings.
* **Causal Masking:** Prevent attention leakage to future tokens using an additive causal upper-triangular mask.
* **Transformer Blocks:** Pass representations sequentially through $N$ identical `TransformerBlock` layers to capture context-aware multi-head attention and non-linear feature projections.
* **Final Layer Normalization (`ln_f`):** Apply layer normalization to the final block's representation for scale and variance stabilization.
* **Logits Projection (Weight Tying):** Project hidden states back to vocabulary logits using the transpose of the input token embedding matrix ($W_{\text{embed}}^T$), reducing total parameter count and improving convergence.

---

### `count_parameters` Method

#### Purpose:
Analytically compute the total learnable parameters directly from layer shapes and dimensional configurations, adhering to test constraints without relying on `sum(p.numel() for p in self.parameters())`.

#### Parameter Formulations:
* **Embeddings:**
  * Token Embeddings: $V \times D$
  * Positional Embeddings: $L_{\max} \times D$
* **Per TransformerBlock ($N$ blocks total):**
  * **MultiHeadAttention:**
    * 4 linear projections ($W_q, W_k, W_v, W_{\text{out}}$) without bias: $4 \times (D \times D)$
  * **FeedForward Network:**
    * $\text{fc}_1$: Weights $(D \times D_{\text{ff}}) + \text{Bias } (D_{\text{ff}})$
    * $\text{fc}_2$: Weights $(D_{\text{ff}} \times D) + \text{Bias } (D)$
  * **Layer Normalizations ($\text{ln}_1, \text{ln}_2$):**
    * 2 learnable vectors ($\gamma, \beta$) per norm: $2 \times (2 \times D) = 4 \times D$
* **Final LayerNorm (`ln_f`):**
  * Scale and shift parameters ($\gamma, \beta$): $2 \times D$


In [13]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=50257, embed_dim=768, num_heads=12,
                 num_layers=12, max_seq_len=1024, ff_dim=3072):
        """
        Full MiniGPT causal language model.

        Args:
            vocab_size (int): Size of the vocabulary. Defaults to 50257.
            embed_dim (int): Hidden dimension size. Defaults to 768.
            num_heads (int): Number of attention heads. Defaults to 12.
            num_layers (int): Number of stacked Transformer blocks. Defaults to 12.
            max_seq_len (int): Maximum sequence context length. Defaults to 1024.
            ff_dim (int): Expansion dimension for the feed-forward network. Defaults to 3072.

        Attributes:
            embedding (Embedding): Joint token and positional embedding layer.
            blocks (nn.ModuleList): Stack of `num_layers` TransformerBlock modules.
            ln_f (LayerNorm): Final normalization applied before the output projection.
            vocab_size (int): Stored vocabulary size.
            embed_dim (int): Stored embedding dimension.
            max_seq_len (int): Stored maximum sequence length.
        """
        super().__init__()
        self.embedding = Embedding(vocab_size, embed_dim, max_seq_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ])
        self.ln_f = LayerNorm(embed_dim)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

    def forward(self, token_ids):
        """
        Forward pass converting token sequences into next-token logits.

        Args:
            token_ids (torch.Tensor): Batch of token index sequences, dtype torch.long.
                Shape: (batch_size, seq_len)

        Returns:
            torch.Tensor: Unnormalized scores over the vocabulary (logits), produced by
                weight tying with the token embedding matrix. A causal mask built from
                `seq_len` must prevent every position from attending to later positions.
                Shape: (batch_size, seq_len, vocab_size)
        """
        B, T = token_ids.shape
        device = token_ids.device

        # Embeddings
        x = self.embedding(token_ids)

        #Causal Mask
        mask = causal_mask(T, dtype=x.dtype, device=device)

        # TransformerBlock
        for block in self.blocks:
            x = block(x, mask=mask)

        # NOrmalizer
        x = self.ln_f(x)

        # 5. Weight Tying for Logits computation (Matrix multiplication with token embedding weights)
        # x: (B, T, embed_dim), token_embed: (vocab_size, embed_dim)
        # Output: (B, T, vocab_size)
        token_weights = self.embedding.token_embed.weight
        logits = torch.matmul(x, token_weights.t())

        return logits

    def count_parameters(self):
        """
        Computes the total number of trainable parameters from the architecture itself.

        Count the token and positional embedding tables, the four attention projections,
        both feed-forward weights and biases, both LayerNorm gains and shifts of every
        block, and the final LayerNorm parameters — deriving each size from the
        hyper-parameters. Do NOT use self.parameters(); the test compares your result
        against it.

        Returns:
            int: Grand total parameter count across all components.
        """
        V = self.vocab_size
        D = self.embed_dim
        M = self.max_seq_len
        num_layers = len(self.blocks)
        # first layer ff_dim
        ff_dim = self.blocks[0].ffn.fc1.out_features

        # 1. Embeddings
        token_emb = V * D
        pos_emb = M * D

        # Per-block parameters
        # Attention: W_q, W_k, W_v, W_out (bias=False)
        attn_params = 4 * (D * D)

        # FeedForward: fc1 (weight + bias) + fc2 (weight + bias)
        ffn_params = (D * ff_dim + ff_dim) + (ff_dim * D + D)

        # Block LayerNorms: ln1 (gamma + beta) + ln2 (gamma + beta)
        block_ln_params = 2 * (2 * D)

        block_total = (attn_params + ffn_params + block_ln_params) * num_layers

        # Final LayerNorm: ln_f (gamma + beta)
        final_ln_params = 2 * D

        total_params = token_emb + pos_emb + block_total + final_ln_params
        return total_params


In [14]:
model = MiniGPT(vocab_size=1000, embed_dim=128, num_heads=4, num_layers=2, max_seq_len=64, ff_dim=512)

# sampel token random
tokens = torch.randint(0, 1000, (2, 16)) # (batch=2, seq_len=16)
logits = model(tokens)

print("Logits shape:", logits.shape)  # torch.Size([2, 16, 1000])

formula_count = model.count_parameters()
actual_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Formula Count: {formula_count:,}")
print(f"Actual Count:  {actual_count:,}")
assert formula_count == actual_count, f"Mismatch: {formula_count} != {actual_count}"
print("Parameter count match verified!")

Logits shape: torch.Size([2, 16, 1000])
Formula Count: 531,968
Actual Count:  531,968
Parameter count match verified!


### Cross-Entropy Loss and Numerical Stability

#### Evaluation Metric (Cross-Entropy):
In autoregressive language model training, the output is a categorical probability distribution over the vocabulary. Cross-entropy quantifies the divergence between predicted token probabilities and target ground truth (one-hot indicator with probability 1 at the target index):

$$\mathcal{L} = -\log P(\text{target})$$

#### Numerical Stability (Log-Sum-Exp Trick):
Direct computation of $\log(\text{softmax}(z))$ is susceptible to numerical overflow from large exponentials ($e^{z_i}$) or underflow from precision degradation.

The Log-Sum-Exp trick stabilizes evaluation by subtracting the row-wise maximum ($M = \max(z)$), ensuring exponentials evaluate within $(-\infty, 0]$:

$$\log\left(\frac{e^{z_k}}{\sum_j e^{z_j}}\right) = (z_k - M) - \log\left(\sum_j e^{z_j - M}\right)$$

#### Differentiability and Autograd Graph Integrity:
All operations must remain native PyTorch tensor operations without casting to Python primitives (e.g., `.item()`, `.numpy()`), preserving the computational graph for valid gradient propagation via `loss.backward()`.

---

### Execution Steps

1. **Dimensional Flattening:**
   Reshape `logits` from $(B, T, V)$ to $(N, V)$ where $N = B \times T$. Flatten `targets` from $(B, T)$ to $(N,)$.

2. **Stable Log-Softmax Computation:**
   * Extract row-wise maxima across vocabulary: `max_logits = logits.max(dim=-1, keepdim=True).values`
   * Center logits: `shifted = logits - max_logits`
   * Compute normalizer: `log_sum_exp = torch.log(torch.sum(torch.exp(shifted), dim=-1, keepdim=True))`
   * Compute log-probabilities: `log_probs = shifted - log_sum_exp`

3. **Target Probability Extraction (NLL):**
   Extract target log-probabilities per token using `torch.gather(log_probs, dim=-1, index=targets.unsqueeze(-1))`.

4. **Batch Reduction:**
   Invert signs and compute the mean across all valid tokens to produce a differentiable scalar loss.


In [15]:
def cross_entropy_loss(logits, targets):
    """
    Computes the average cross-entropy loss over a batch of sequences.

    Must be implemented with a numerically stable log-softmax written by hand, and must
    stay differentiable: the returned tensor is what `.backward()` is called on during
    training, so do not detach it or convert it to a Python float.

    Args:
        logits (torch.Tensor): Model output logits before softmax.
            Shape: (batch_size, seq_len, vocab_size)
        targets (torch.Tensor): Ground-truth target token indices, dtype torch.long.
            Shape: (batch_size, seq_len)

    Returns:
        torch.Tensor: Scalar (0-dimensional) loss tensor, averaged over all
            batch_size * seq_len positions.
            Shape: ()
    """
    #Flatten logits: (B, T, V) -> (N, V) | targets: (B, T) -> (N,)
    N_V = logits.view(-1, logits.size(-1))
    flat_targets = targets.view(-1, 1)

    #Numerically stable Log-Softmax
    max_logits = N_V.max(dim=-1, keepdim=True).values
    shifted = N_V - max_logits
    log_sum_exp = torch.log(torch.sum(torch.exp(shifted), dim=-1, keepdim=True))
    log_probs = shifted - log_sum_exp

    # Extract log-probability of target indices: (N, 1)
    target_log_probs = torch.gather(log_probs, dim=-1, index=flat_targets)

    # Negative Log-Likelihood and average over all positions
    loss = -target_log_probs.mean()

    return loss


In [16]:
#test
B, T, V = 2, 4, 10
dummy_logits = torch.randn(B, T, V, requires_grad=True)
dummy_targets = torch.randint(0, V, (B, T))


my_loss = cross_entropy_loss(dummy_logits, dummy_targets)

torch_loss = torch.nn.functional.cross_entropy(
    dummy_logits.view(-1, V), dummy_targets.view(-1))

print(f"Custom Loss: {my_loss.item():.6f}")
print(f"Torch Loss:  {torch_loss.item():.6f}")
assert torch.allclose(my_loss, torch_loss, atol=1e-5), "Losses do not match!"

#
my_loss.backward()
assert dummy_logits.grad is not None, "Grad is None! Loss is not differentiable."
print("Backward pass verified successfully!")


Custom Loss: 3.155769
Torch Loss:  3.155769
Backward pass verified successfully!


### Autoregressive Generation & Sampling Mechanics

#### 1. Autoregressive Generation
Autoregressive language models generate text token-by-token. At each forward step, the predicted token is appended to the current sequence context and passed back into the model to predict the subsequent token.

#### 2. Temperature Sampling
Greedy decoding (always selecting the token with the highest logit) often results in repetitive, deterministic loops. Instead, stochastic sampling using `torch.multinomial` samples tokens according to their predicted probability distribution.

The **Temperature** parameter ($T$) controls distribution sharpness:

$$P(y = v) = \text{Softmax}\left(\frac{z_v}{T}\right) = \frac{\exp(z_v / T)}{\sum_j \exp(z_j / T)}$$

* **Lower Temperature ($T \to 0$):** Concentrates probability mass onto the highest-scoring tokens, yielding conservative, predictable text.
* **Higher Temperature ($T > 1$):** Flattens the distribution toward uniform probability, increasing lexical diversity at the expense of coherence.

#### 3. Context Window Truncation
The input sequence length cannot exceed the model's fixed positional capacity (`max_seq_len`). When sequence length exceeds this threshold, the context is truncated to preserve only the most recent `max_seq_len` tokens.

#### 4. Inference Context (`torch.no_grad()`)
Text generation disables autograd tracking via `torch.no_grad()`, reducing GPU VRAM allocation and eliminating backward-pass computation overhead.

---

### Generation Pipeline Steps

1. **Input Tensor Setup:**
   Convert `prompt_tokens` into a 2D integer tensor of shape `(1, seq_len)` placed on the model's target compute `device`.

2. **Autoregressive Generation Loop ($N$ tokens):**
   * **Context Truncation:** Crop context window to respect sequence limits: `idx_cond = idx[:, -model.max_seq_len:]`
   * **Forward Pass:** Evaluate unnormalized predictions: `logits = model(idx_cond)` with shape `(1, T, vocab_size)`.
   * **Isolate Last Token:** Slice the active prediction step: `logits = logits[:, -1, :]` with shape `(1, vocab_size)`.
   * **Scale & Softmax:** Apply temperature scaling and compute probabilities: `probs = F.softmax(logits / temperature, dim=-1)`.
   * **Stochastic Sampling:** Sample the next token index: `idx_next = torch.multinomial(probs, num_samples=1)` with shape `(1, 1)`.
   * **Sequence Concatenation:** Append sampled token to accumulated context: `idx = torch.cat((idx, idx_next), dim=1)`.

3. **Output Serialization:**
   Flatten and cast the resulting sequence tensor back to a Python list of token IDs.


In [17]:
@torch.no_grad()
def generate(model, prompt_tokens, max_new_tokens=100, temperature=0.8):
    """
    Autoregressively generates new tokens from a prompt using temperature sampling.

    Runs without gradient tracking. Sampling must use torch.multinomial, so that seeding
    with torch.manual_seed makes the output reproducible.

    Args:
        model (MiniGPT): The language model instance.
        prompt_tokens (list[int]): Initial prompt token IDs.
        max_new_tokens (int): Number of new tokens to generate. Defaults to 100.
        temperature (float): Divisor applied to the logits before the softmax. Lower
            values sharpen the distribution, higher values flatten it. Defaults to 0.8.

    Returns:
        list[int]: The prompt followed by the generated tokens, of total length
            len(prompt_tokens) + max_new_tokens. The context fed to the model at each
            step must be truncated to the model's maximum sequence length.
    """
    # Eval mode for Start assessment
    model.eval()

    # Extracting the device model from the first available parameter
    device = next(model.parameters()).device

    # change to tensor (1, seq_len)
    idx = torch.tensor(prompt_tokens, dtype=torch.long, device=device).unsqueeze(0)

    # loop automatic backforward
    for _ in range(max_new_tokens):
        # split to max_seq_len
        idx_cond = idx if idx.size(1) <= model.max_seq_len else idx[:, -model.max_seq_len:]

        # forward
        logits = model(idx_cond)

        # Extracting logits for the last token
        logits = logits[:, -1, :] / temperature

        # Softmax
        probs = torch.nn.functional.softmax(logits, dim=-1)

        # Sampling from a probability distribution while maintaining reproducibility
        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return idx[0].tolist()

In [18]:
# test generate
torch.manual_seed(42)

# sampel 
test_model = MiniGPT(vocab_size=100, embed_dim=32, num_heads=2, num_layers=2, max_seq_len=20, ff_dim=64)

prompt = [1, 2, 3, 4]
max_new = 10
out_tokens = generate(test_model, prompt, max_new_tokens=max_new, temperature=0.8)

print("Generated tokens:", out_tokens)
print(f"Initial length: {len(prompt)}, Output length: {len(out_tokens)}")

assert len(out_tokens) == len(prompt) + max_new, "Output length mismatch!"
assert out_tokens[:len(prompt)] == prompt, "Prompt tokens were corrupted!"
print("Generation logic verified successfully!")

Generated tokens: [1, 2, 3, 4, 69, 31, 25, 81, 22, 98, 31, 63, 16, 34]
Initial length: 4, Output length: 14
Generation logic verified successfully!
